In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')


In [3]:
df = pd.read_csv(r'd:\Projects\credit-risk-network\data\raw\cs-training.csv\cs-training.csv', index_col=0)
print("Shape:", df.shape)
df.head()

Shape: (150000, 11)


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [4]:
# Impute MonthlyIncome with median grouped by age bracket
df['age_bracket'] = pd.cut(df['age'], bins=[0, 25, 35, 45, 55, 65, 100], 
                            labels=['<25', '25-35', '35-45', '45-55', '55-65', '65+'])

df['MonthlyIncome'] = df.groupby('age_bracket', observed=True)['MonthlyIncome']\
                        .transform(lambda x: x.fillna(x.median()))

# If any still missing, fill with overall median
df['MonthlyIncome'].fillna(df['MonthlyIncome'].median(), inplace=True)

# Impute NumberOfDependents with 0
df['NumberOfDependents'].fillna(0, inplace=True)

# Drop age_bracket helper column
df.drop('age_bracket', axis=1, inplace=True)

print("Missing values after imputation:")
print(df.isnull().sum())

Missing values after imputation:
SeriousDlqin2yrs                           0
RevolvingUtilizationOfUnsecuredLines       0
age                                        0
NumberOfTime30-59DaysPastDueNotWorse       0
DebtRatio                                  0
MonthlyIncome                             14
NumberOfOpenCreditLinesAndLoans            0
NumberOfTimes90DaysLate                    0
NumberRealEstateLoansOrLines               0
NumberOfTime60-89DaysPastDueNotWorse       0
NumberOfDependents                      3924
dtype: int64


In [5]:
# Fix remaining missing values
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(0)

print("Missing values after fix:")
print(df.isnull().sum())
print("\nAll zeros:", df.isnull().sum().sum())

Missing values after fix:
SeriousDlqin2yrs                        0
RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
dtype: int64

All zeros: 0


In [6]:
# Cap extreme outliers using 99th percentile
cols_to_cap = [
    'RevolvingUtilizationOfUnsecuredLines',
    'DebtRatio',
    'MonthlyIncome',
    'NumberOfOpenCreditLinesAndLoans',
    'NumberRealEstateLoansOrLines',
    'NumberOfDependents'
]

for col in cols_to_cap:
    cap = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=cap)

# Remove invalid ages
df = df[df['age'] > 0]

print("Shape after outlier removal:", df.shape)
print("Default rate preserved:", df['SeriousDlqin2yrs'].mean().round(3))

Shape after outlier removal: (149999, 11)
Default rate preserved: 0.067


In [7]:
# Separate features and target
X = df.drop('SeriousDlqin2yrs', axis=1)
y = df['SeriousDlqin2yrs']

# Scale features
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Train-test split (stratified to preserve default rate)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("Default rate in train:", y_train.mean().round(3))
print("Default rate in test:", y_test.mean().round(3))

Training set: (119999, 10)
Test set: (30000, 10)
Default rate in train: 0.067
Default rate in test: 0.067


In [8]:
import os

# Save processed data
X_train.to_csv(r'd:\Projects\credit-risk-network\data\processed\X_train.csv', index=False)
X_test.to_csv(r'd:\Projects\credit-risk-network\data\processed\X_test.csv', index=False)
y_train.to_csv(r'd:\Projects\credit-risk-network\data\processed\y_train.csv', index=False)
y_test.to_csv(r'd:\Projects\credit-risk-network\data\processed\y_test.csv', index=False)

# Save scaler columns for later use
pd.DataFrame(X.columns, columns=['features']).to_csv(
    r'd:\Projects\credit-risk-network\data\processed\feature_names.csv', index=False)

print("=== PREPROCESSING COMPLETE ===")
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")
print("\nFiles saved to data/processed/")
print("Ready for Notebook 3: Baseline Models")

=== PREPROCESSING COMPLETE ===
X_train: (119999, 10)
X_test: (30000, 10)
y_train: (119999,)
y_test: (30000,)

Files saved to data/processed/
Ready for Notebook 3: Baseline Models
